In [ ]:
#| default_exp html

# html

> Generate a self-contained HTML reader for a rendered comic.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path

from manhualizer.models import ComicOutput, Storyboard

In [ ]:
#| export
_CSS = """
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

body {
    background: #0d0d0d;
    color: #e0e0e0;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    display: flex;
    flex-direction: column;
    align-items: center;
    padding: 2rem 1rem 4rem;
    gap: 0;
}

header {
    width: 100%;
    max-width: 620px;
    text-align: center;
    padding-bottom: 2rem;
    border-bottom: 1px solid #2a2a2a;
    margin-bottom: 2rem;
}

header h1 {
    font-size: 1.4rem;
    font-weight: 700;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    color: #f0f0f0;
}

header p {
    font-size: 0.75rem;
    color: #555;
    margin-top: 0.4rem;
    letter-spacing: 0.05em;
}

.scene-title {
    width: 100%;
    max-width: 620px;
    font-size: 0.7rem;
    font-weight: 600;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    color: #444;
    padding: 1.8rem 0 0.6rem;
    border-top: 1px solid #1e1e1e;
    margin-top: 1rem;
}

.scene-title.first {
    border-top: none;
    margin-top: 0;
    padding-top: 0;
}

.panel {
    width: 100%;
    max-width: 620px;
    display: block;
}

.panel img {
    width: 100%;
    height: auto;
    display: block;
}

footer {
    margin-top: 3rem;
    font-size: 0.65rem;
    color: #333;
    letter-spacing: 0.06em;
}
"""


def generate_html_viewer(
    output: ComicOutput,
    storyboard: Storyboard,
    title: str = "",
) -> Path:
    """Write index.html into output_dir and return its path.

    The page is a self-contained vertical-scroll reader styled for manhua.
    Images are referenced by relative path so the directory is portable.
    """
    panels_dir = output.output_dir / "panels"
    panel_map = (
        {p.stem: p for p in sorted(panels_dir.glob("panel_*"))}
        if panels_dir.exists() else {}
    )

    rows: list[str] = []
    for scene_idx, scene in enumerate(storyboard.scenes):
        scene_label = scene.title or f"Scene {scene.scene_id}"
        first_cls = ' class="scene-title first"' if scene_idx == 0 else ' class="scene-title"'
        rows.append(f'  <p{first_cls}>{_esc(scene_label)}</p>')
        for panel in scene.panels:
            key = f"panel_{panel.panel_number:04d}"
            img_path = panel_map.get(key)
            if img_path is None:
                continue
            rel = img_path.relative_to(output.output_dir)
            rows.append(
                f'  <div class="panel">'
                f'<img src="{rel}" alt="Panel {panel.panel_number}" loading="lazy">'
                f'</div>'
            )

    panel_count = sum(1 for r in rows if 'class="panel"' in r)
    n_scenes = len(storyboard.scenes)
    subtitle = (
        f"{n_scenes} scene{'s' if n_scenes != 1 else ''}"
        f" &middot; {panel_count} panel{'s' if panel_count != 1 else ''}"
    )
    display_title = _esc(title) if title else "Comic"

    html = (
        '<!DOCTYPE html>\n'
        '<html lang="en">\n'
        '<head>\n'
        '  <meta charset="UTF-8">\n'
        '  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n'
        f'  <title>{display_title}</title>\n'
        f'  <style>{_CSS}</style>\n'
        '</head>\n'
        '<body>\n'
        '  <header>\n'
        f'    <h1>{display_title}</h1>\n'
        f'    <p>{subtitle}</p>\n'
        '  </header>\n'
        + '\n'.join(rows) + '\n'
        '  <footer>generated by manhualizer</footer>\n'
        '</body>\n'
        '</html>'
    )

    out = output.output_dir / "index.html"
    out.write_text(html, encoding="utf-8")
    return out


def _esc(s: str) -> str:
    return (
        s.replace("&", "&amp;")
         .replace("<", "&lt;")
         .replace(">", "&gt;")
         .replace('"', "&quot;")
    )

## Tests

In [ ]:
import tempfile
from pathlib import Path
from manhualizer.html import generate_html_viewer, _esc
from manhualizer.models import ComicOutput, Storyboard

# _esc
assert _esc('<b>"hi"</b>') == '&lt;b&gt;&quot;hi&quot;&lt;/b&gt;'

# generate_html_viewer with fixture storyboard
# nbdev-test runs from nbs/ directory, so go up one level
fixture = Path("..") / "tests" / "fixtures"
storyboard = Storyboard.model_validate_json((fixture / "storyboard.json").read_text())

with tempfile.TemporaryDirectory() as tmp:
    out_dir = Path(tmp)
    panels_dir = out_dir / "panels"
    panels_dir.mkdir()
    for i in range(1, 3):
        (panels_dir / f"panel_{i:04d}.png").write_bytes(b"")

    comic = ComicOutput(
        output_dir=out_dir,
        analysis_path=out_dir / "analysis.json",
        storyboard_path=out_dir / "storyboard.json",
    )
    html_path = generate_html_viewer(comic, storyboard, title="The Dragon's Gift")
    assert html_path.exists()
    content = html_path.read_text()
    assert "The Dragon" in content
    assert html_path.name == "index.html"
    assert "panels/panel_" in content
    assert "scene-title" in content
    assert "generated by manhualizer" in content

print("HTML viewer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()